# Phase II Data Quality Analysis

This notebook evaluates the quality and structure of the balanced Google Play Store and Apple App Store review datasets before deeper exploratory data analysis.

In [3]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Find project root whether the notebook runs from the root
# or from the notebooks folder
ROOT = Path.cwd()

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

APPLE_PATH = ROOT / "data" / "processed" / "apple_store_analysis.csv"
GOOGLE_PATH = ROOT / "data" / "processed" / "google_play_analysis.csv"

apple = pd.read_csv(APPLE_PATH)
google = pd.read_csv(GOOGLE_PATH)

print("Apple rows:", len(apple))
print("Google rows:", len(google))

Apple rows: 13599
Google rows: 13599


## 1. Dataset Overview

In [4]:
print("Apple dataset shape:")
print(apple.shape)

print("\nGoogle Play dataset shape:")
print(google.shape)

print("\nApple columns:")
print(apple.columns.tolist())

print("\nGoogle Play columns:")
print(google.columns.tolist())

Apple dataset shape:
(13599, 14)

Google Play dataset shape:
(13599, 15)

Apple columns:
['platform', 'app_name', 'app_id', 'category', 'review_id', 'review_title', 'review_text', 'rating', 'review_date', 'app_version', 'reviewer_name', 'helpful_count', 'source_page', 'collection_timestamp']

Google Play columns:
['platform', 'app_name', 'app_id', 'category', 'review_id', 'review_text', 'rating', 'review_date', 'app_version', 'developer_response', 'developer_response_date', 'reviewer_name', 'helpful_count', 'source_batch', 'collection_timestamp']


## 2. Missing Values

In [5]:
apple_missing = pd.DataFrame({
    "missing_count": apple.isnull().sum(),
    "missing_percent": (apple.isnull().mean() * 100).round(2)
})

google_missing = pd.DataFrame({
    "missing_count": google.isnull().sum(),
    "missing_percent": (google.isnull().mean() * 100).round(2)
})

print("Apple Missing Values:")
display(apple_missing)

print("\nGoogle Play Missing Values:")
display(google_missing)

Apple Missing Values:


,missing_count,missing_percent
platform,0,0.00
app_name,0,0.00
app_id,0,0.00
category,0,0.00
review_id,0,0.00
review_title,2,0.01
review_text,1,0.01
rating,0,0.00
review_date,0,0.00
app_version,0,0.00



Google Play Missing Values:


,missing_count,missing_percent
platform,0,0.00
app_name,0,0.00
app_id,0,0.00
category,0,0.00
review_id,0,0.00
review_text,1,0.01
rating,0,0.00
review_date,0,0.00
app_version,2190,16.10
developer_response,10895,80.12


### Initial Findings

- Apple App Store data is almost complete across the observed fields. Only 1 review text and 2 review titles are missing out of 13,599 reviews.
- Google Play core fields such as review ID, rating, review date, and reviewer name are highly complete.
- Google Play app-version metadata is less complete, with 2,190 missing values (16.10%).
- Developer responses are much sparser on Google Play: 10,895 reviews (80.12%) do not contain a developer response.
- The two platforms therefore differ meaningfully in metadata availability, which should be considered when designing downstream analyses.

## 3. Duplicate and Repeated Review Checks

In [6]:
duplicate_summary = pd.DataFrame({
    "metric": [
        "Duplicate review IDs",
        "Repeated review text"
    ],
    "Apple App Store": [
        apple["review_id"].duplicated().sum(),
        apple["review_text"].duplicated().sum()
    ],
    "Google Play Store": [
        google["review_id"].duplicated().sum(),
        google["review_text"].duplicated().sum()
    ]
})

display(duplicate_summary)

,metric,Apple App Store,Google Play Store
0,Duplicate review IDs,0,0
1,Repeated review text,454,2319


In [7]:
apple_repeated_pct = (
    apple["review_text"].duplicated().mean() * 100
)

google_repeated_pct = (
    google["review_text"].duplicated().mean() * 100
)

print(
    f"Apple repeated-text rate: "
    f"{apple_repeated_pct:.2f}%"
)

print(
    f"Google Play repeated-text rate: "
    f"{google_repeated_pct:.2f}%"
)

Apple repeated-text rate: 3.34%
Google Play repeated-text rate: 17.05%


### Initial Findings

- No duplicate review IDs remain in either processed dataset.
- Repeated review text is substantially more common on Google Play Store than on Apple App Store.
- Approximately 17.05% of Google Play reviews repeat text that appears elsewhere in the dataset, compared with 3.34% on Apple App Store.
- Repeated text does not necessarily indicate duplicate records, since different users may independently submit short comments such as "Good", "Great", or "Love it".
- Repeated review text will therefore be retained for the main EDA and examined further as a data-quality characteristic rather than automatically removed.

## 4. Rating Distribution

In [ ]:
apple_rating = (
    apple["rating"]
    .value_counts()
    .sort_index()
)

google_rating = (
    google["rating"]
    .value_counts()
    .sort_index()
)

rating_summary = pd.DataFrame({
    "Apple App Store": apple_rating,
    "Google Play Store": google_rating
})

display(rating_summary)